In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
import numpy as np

results_path = "../results/"
h3_path = os.path.join(results_path, "h3/*.csv")

df_smax_120 = pd.read_csv(os.path.join(results_path, "solution_quality_test_smax_1.2.csv"))
df_smax_125 = pd.read_csv(os.path.join(results_path, "solution_quality_test_smax_1.25.csv"))
df_smax_inf = pd.read_csv(os.path.join(results_path, "solution_quality_test_smax_inf.csv"))

h3_result_files = glob.glob(h3_path)
df_h3 = pd.concat([pd.read_csv(f).assign(Instance=os.path.basename(f).replace('_h3.csv', '')) for f in h3_result_files])

sns.set_theme(style="whitegrid")

In [2]:
stats = df_smax_125.groupby(['Instance', 'Algorithm'])['Distance'].agg(['min', 'mean', 'std']).reset_index()
stats.columns = ['Instance', 'Algorithm', 'Best', 'Average', 'Std_Dev']

display(stats.sort_values(by=['Instance', 'Best']))

,Instance,Algorithm,Best,Average,Std_Dev
3,A-n32-k5.vrp,MMAS,791.569861,792.424961,0.717901
0,A-n32-k5.vrp,ASRank,799.096009,802.822654,3.182566
1,A-n32-k5.vrp,BasicAntSystem,811.971674,819.259273,5.854696
2,A-n32-k5.vrp,Greedy,1064.656130,1064.656130,NaN
7,A-n33-k6.vrp,MMAS,819.951556,877.944068,36.663775
4,A-n33-k6.vrp,ASRank,858.102253,921.204339,35.082923
5,A-n33-k6.vrp,BasicAntSystem,891.286997,930.150615,31.435002
6,A-n33-k6.vrp,Greedy,971.427572,971.427572,NaN
11,A-n36-k5.vrp,MMAS,870.372450,894.389899,16.930158
8,A-n36-k5.vrp,ASRank,890.552522,958.387108,43.624122


In [ ]:
def plot_instance_boxplots(df, instance_name):
    plt.figure(figsize=(10, 6))
    data = df[df['Instance'] == instance_name].where(df['Algorithm'].isin(['MMAS', 'ASRank', 'BasicAntSystem']))
    sns.boxplot(data=data, x='Algorithm', y='Distance', palette='Set2', hue="Algorithm")
    plt.title(f"Variation of results for instance: {instance_name} (S_max 1.25)")
    plt.show()

unique_instances = df_smax_125['Instance'].unique()
for inst in unique_instances:
    plot_instance_boxplots(df_smax_125, inst)

In [ ]:
def plot_convergence(df_h3, instance_name):
    plt.figure(figsize=(12, 6))
    
    # Filter data for the specific instance
    inst_data = df_h3[df_h3['Instance'] == instance_name]

    # Draw line plot with standard deviation error bars
    sns.lineplot(data=inst_data, x='Iteration', y='Best_Cost', hue='Algorithm', errorbar='sd')
    
    # Draw line for 20% of iterations
    plt.axvline(x=40, color='red', linestyle='--', label='20% of iterations')
    
    plt.title(f"Convergence Plot: {instance_name}")
    plt.ylabel("Best Found Cost")
    plt.legend()
    plt.show()

for inst in df_h3['Instance'].unique():
    plot_convergence(df_h3, inst)

In [8]:
h1_data = df_smax_125.groupby(['Instance', 'Algorithm'])['Distance'].mean().unstack()

h1_data['Threshold_85pct'] = h1_data['Greedy'] * 0.85

for algo in ['BasicAntSystem', 'MMAS', 'ASRank']:
    h1_data[f'{algo}_Satisfies_H1'] = h1_data[algo] <= h1_data['Threshold_85pct']

verification_h1 = h1_data[[algo + '_Satisfies_H1' for algo in ['BasicAntSystem', 'MMAS', 'ASRank']]]
display(verification_h1)

success_rate = verification_h1.mean() * 100
print(f"Percentage of instances satisfying H1:\n{success_rate}")
print(f"Decision: {'CONFIRMED' if (success_rate >= 90).all() else 'REJECTED'}")

Algorithm,BasicAntSystem_Satisfies_H1,MMAS_Satisfies_H1,ASRank_Satisfies_H1
Instance,,,
A-n32-k5.vrp,True,True,True
A-n33-k6.vrp,False,False,False
A-n36-k5.vrp,False,False,False
A-n37-k6.vrp,False,True,False
A-n39-k5.vrp,True,True,True
A-n45-k7.vrp,False,False,False
A-n55-k9.vrp,False,False,False
A-n60-k9.vrp,True,True,True
A-n63-k10.vrp,False,False,False


Percentage of instances satisfying H1:
Algorithm
BasicAntSystem_Satisfies_H1    36.363636
MMAS_Satisfies_H1              45.454545
ASRank_Satisfies_H1            36.363636
dtype: float64
Decision: REJECTED


In [ ]:
h2_std = df_smax_125[df_smax_125['Algorithm'] != 'Greedy'].groupby(['Instance', 'Algorithm'])['Distance'].std().unstack()

h2_std['Min_STD'] = h2_std.min(axis=1)
h2_std['Is_MMAS_Most_Stable'] = h2_std['MMAS'] == h2_std['Min_STD']

display(h2_std)

total_mmas_wins = h2_std['Is_MMAS_Most_Stable'].mean() * 100

print(f"MMAS was most stable in {total_mmas_wins:.2f}% of all instances.")
print(f"Decision (threshold 80%): {'CONFIRMED' if total_mmas_wins >= 80 else 'REJECTED'}")

Algorithm,ASRank,BasicAntSystem,MMAS,Min_STD,Is_MMAS_Most_Stable
Instance,,,,,
A-n32-k5.vrp,3.182566,5.854696,0.717901,0.717901,True
A-n33-k6.vrp,35.082923,31.435002,36.663775,31.435002,False
A-n36-k5.vrp,43.624122,35.551075,16.930158,16.930158,True
A-n37-k6.vrp,54.049488,46.535633,22.984452,22.984452,True
A-n39-k5.vrp,0.000000,0.000000,0.000000,0.000000,True
A-n45-k7.vrp,27.983010,25.416758,5.724212,5.724212,True
A-n55-k9.vrp,27.466478,23.932133,15.622510,15.622510,True
A-n60-k9.vrp,51.282653,52.595216,15.820495,15.820495,True
A-n63-k10.vrp,66.188069,56.090188,44.641409,44.641409,True


MMAS was most stable in 81.82% of all instances.
Decision (threshold 80%): CONFIRMED
